In [1]:
import torch
import torchvision.transforms as transforms
import pandas as pd 
import numpy as np 
import random
import os
from loguru import logger
import rasterio
from matplotlib import pyplot as plt
import argparse
import cv2
import imageio

from models.get_model import get_model

In [2]:

def prepare_inp_data(features, nodata_val=-9999):
    feats_tmp = np.where(features==nodata_val, np.nanmax(features), features)
    feats_min = np.nanmin(feats_tmp)
    feats_tmp = np.where(features==nodata_val, feats_min, features)    # replace nodata values with minimum
    feats_tmp = np.where(np.isnan(feats_tmp), feats_min, feats_tmp)    # replace nan values with minimum
    feats_tmp = np.nan_to_num(feats_tmp, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    data_transforms = transforms.Compose([
        transforms.ToTensor(),
    ])
    image = feats_tmp.astype(np.float32)
    image = data_transforms(image)
    # Normalize data
    if (torch.max(image)-torch.min(image)):
        image = image - torch.min(image)
        image = image / torch.maximum(torch.max(image),torch.tensor(1))
    else:
        image = np.zeros_like(image)
    image = image.type(torch.FloatTensor)
    return image

In [3]:
EPS = 1e-7
def compute_mask_metrics(pred, target, thresh=0.5):
    thresh_pred = np.where(pred > thresh, 1., 0.)
    rec, _, TP_FN = get_recall(thresh_pred, target)
    prec, TP, TP_FP = get_precision(thresh_pred, target)
    f1 = get_f1(thresh_pred, target)

    return rec, prec, f1, TP, TP_FN, TP_FP

def get_recall(y_pred, y_true):
    TP = np.sum(np.round(np.clip(y_true * y_pred, 0, 1)))
    TP_FN = np.sum(np.round(np.clip(y_true, 0, 1)))
    recall = TP / (TP_FN + EPS)
    return recall, TP, TP_FN

def get_precision(y_pred, y_true):
    TP = np.sum(np.round(np.clip(y_true * y_pred, 0, 1)))
    TP_FP = np.sum(np.round(np.clip(y_pred, 0, 1)))
    precision = TP / (TP_FP + EPS)
    return precision, TP, TP_FP

def get_f1(y_pred, y_true):
    precision, _, _ = get_recall(y_pred, y_true)
    recall, _, _ = get_precision(y_pred, y_true)
    return 2 * ((precision * recall) / (precision + recall + EPS))

In [4]:
class Namespace:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

backbone = "mobilenetv3" # deeplabv3p, mobilenetv3, segnet
is_distrib = True
ckpt_path = f"ckpts/{backbone}_distrib.pth.tar"
opt = Namespace(
        ckpt_path=ckpt_path,
        backbone=f'{backbone}',
        head=f'{backbone}_head',
        method='vanilla', 
        tasks=['water_mask', 'cloudshadow_mask', 'cloud_mask', 'snowice_mask', 'sun_mask'], 
    )


num_inp_feats = 6   # number of channels in input
tasks_outputs = {
    "water_mask": 1,
    "cloudshadow_mask": 1,
    "cloud_mask": 1,
    "snowice_mask": 1,
    "sun_mask": 1,
}
model = get_model(opt, tasks_outputs=tasks_outputs, num_inp_feats=num_inp_feats)

logger.debug(f"Loading weights from {ckpt_path}")
checkpoint = torch.load(ckpt_path, map_location=torch.device('cpu'))
if is_distrib:
    new_ckpt = {k.split("module.")[-1]:v for k,v in checkpoint["state_dict"].items()}
    checkpoint["state_dict"] = new_ckpt
tmp = model.load_state_dict(checkpoint["state_dict"], strict=True)
logger.debug(f"After loading ckpt: {tmp}")
logger.debug(f"Checkpoint epoch: {checkpoint['epoch']}. best_perf: {checkpoint['best_performance']}")

2024-02-07 19:19:40.566 | DEBUG    | models.get_model:get_model:45 - backbone_channels: 128
2024-02-07 19:19:40.574 | DEBUG    | __main__:<module>:27 - Loading weights from ckpts/mobilenetv3_distrib.pth.tar
2024-02-07 19:19:40.732 | DEBUG    | __main__:<module>:33 - After loading ckpt: <All keys matched successfully>
2024-02-07 19:19:40.733 | DEBUG    | __main__:<module>:34 - Checkpoint epoch: 58. best_perf: 0.03457197898533195


In [5]:

if backbone == "deeplabv3p":
    optim_threshes = {     # for DeepLabv3+
        "water_mask": 0.2,
        "cloudshadow_mask": 0.2,
        "cloud_mask": 0.3,
        "snowice_mask": 0.2,
        'sun_mask': 0.3,    # for 261k dataset
    }
elif backbone == "mobilenetv3":
    optim_threshes = {     # for MobileNet
        "water_mask": 0.2,
        "cloudshadow_mask": 0.2,
        "cloud_mask": 0.3,
        "snowice_mask": 0.2,
        'sun_mask': 0.3,
    }
elif backbone == "segnet":
    optim_threshes = {     # for SegNet
        "water_mask": 0.2,
        "cloudshadow_mask": 0.2,
        "cloud_mask": 0.3,
        "snowice_mask": 0.2,
        'sun_mask': 0.5,
    }
model.eval()
print()

In [6]:
# Find samples where model is performing better than Fmask (i.e., higher F1 score)

idxs = []
results = []
for i in range(10095):
    if i <= 7893:
        continue
    fp = f"colorado_data/{i:06d}.npy"
    with open(fp, "rb") as f:
        npzfile = np.load(f)
        input_data = npzfile["input"]  # (6, 512, 512) (C,H,W)
        output_data = npzfile["output"]  # (1, 512, 512) (C,H,W)
        fmask_data = npzfile["fmask"]
        sat_type = npzfile["satellite"]
    input_data = np.transpose(input_data, (1,2,0))  # (512, 512, 6) (H,W,C)
    inp_data = prepare_inp_data(input_data)   # size: (6,512,512)
    inp_data = torch.unsqueeze(inp_data, dim=0) # to have batch size of 1
    with torch.no_grad():
        test_pred, feat = model(inp_data, feat=True)
    masks = {}
    for t in tasks_outputs.keys():
        pred_img = test_pred[t][0,:,:].detach().cpu().numpy()
        thresh = optim_threshes[t]
        masks[t] = (pred_img > thresh).astype(int).squeeze()

    # 1=open water; 2=partial water; 252: snow/ice; 253: Cloud/Cloud Shadow and adjacent to cloud/cloud shadow; 254: Ocean Masked -
    tmp = np.where(output_data==1, 1, 0) + np.where(output_data==2, 1, 0) + np.where(output_data==254, 1, 0)
    water_gt = np.where(tmp>0, 1, 0)
    water_gt = water_gt.squeeze()

    fmask_water_init = (fmask_data>>5)&1     # Bit 5 is water
    fmask_snowice = (fmask_data>>4)&1
    fmask_cloudshadow = (fmask_data>>3)&1
    fmask_cloud = (fmask_data>>1)&1
    fmask_water = (fmask_water_init * (1-fmask_snowice) * (1-fmask_cloudshadow) * (1-fmask_cloud)).squeeze()

    water_mask = masks["water_mask"]
    cloudshadow_mask = masks["cloudshadow_mask"]
    cloud_mask = masks["cloud_mask"]
    snowice_mask = masks["snowice_mask"]
    sun_mask = masks["sun_mask"]
    filtered_water_mask = water_mask * (1-cloudshadow_mask) * (1-cloud_mask) * (1-snowice_mask) * sun_mask
    filtered_water_mask = filtered_water_mask.squeeze()

    _,_,_, model_TP, model_TP_FN, model_TP_FP = compute_mask_metrics(filtered_water_mask, water_gt, thresh=0.5)
    _,_,_, fmask_TP, fmask_TP_FN, fmask_TP_FP = compute_mask_metrics(fmask_water, water_gt, thresh=0.5)
    model_recall = model_TP / (model_TP_FN + EPS)
    model_precision = model_TP / (model_TP_FP + EPS)
    model_f1 = 2 * ((model_precision * model_recall) / (model_precision + model_recall + EPS))

    fmask_recall = fmask_TP / (fmask_TP_FN + EPS)
    fmask_precision = fmask_TP / (fmask_TP_FP + EPS)
    fmask_f1 = 2 * ((fmask_precision * fmask_recall) / (fmask_precision + fmask_recall + EPS))

    elev_shadow_pixels = np.sum(1-sun_mask)
    is_model_better = (model_f1 > fmask_f1)
    logger.debug(f"i: {i}, model: {model_f1}. fmask: {fmask_f1}, elev_shadow_pixels: {elev_shadow_pixels}, is_model_better: {is_model_better}")
    results.append([i, model_f1, model_recall, model_precision, fmask_f1, fmask_recall, fmask_precision, elev_shadow_pixels, is_model_better])

    df = pd.DataFrame(results, columns=["i", "model_f1", "model_recall", "model_precision", "fmask_f1", "fmask_recall", "fmask_precision", "elev_shadow_pixels", "is_model_better"])
    df.to_csv(f"colorado_results_{backbone}.csv", index=False)
    if model_f1 > fmask_f1 and (elev_shadow_pixels>0):
        idxs.append(i)
        logger.info(f"i: {i}, model: {model_f1}. fmask: {fmask_f1}, elev_shadow_pixels: {elev_shadow_pixels}")

2024-02-07 19:19:42.862 | DEBUG    | __main__:<module>:57 - i: 7894, model: 0.017060364989477547. fmask: 0.20950320926632796, elev_shadow_pixels: 0, is_model_better: False
2024-02-07 19:19:44.733 | DEBUG    | __main__:<module>:57 - i: 7895, model: 0.0. fmask: 0.25263154040443797, elev_shadow_pixels: 0, is_model_better: False
2024-02-07 19:19:47.444 | DEBUG    | __main__:<module>:57 - i: 7896, model: 0.5903083279163219. fmask: 0.6297871900479884, elev_shadow_pixels: 0, is_model_better: False
2024-02-07 19:19:50.425 | DEBUG    | __main__:<module>:57 - i: 7897, model: 0.0. fmask: 0.21276592630149838, elev_shadow_pixels: 0, is_model_better: False
2024-02-07 19:19:54.070 | DEBUG    | __main__:<module>:57 - i: 7898, model: 0.6994082371778328. fmask: 0.9087241054672255, elev_shadow_pixels: 0, is_model_better: False
2024-02-07 19:19:57.401 | DEBUG    | __main__:<module>:57 - i: 7899, model: 0.42679896541201945. fmask: 0.4491978238997427, elev_shadow_pixels: 0, is_model_better: False
2024-02-07

## Generate visualizations for select samples

In [7]:
# samp_idxs = [303, ]
# selected = [83, 149, 137, 150, 151, 152, 153, 154, 188, 191, 196, 198, 200, 228, 234]
selected = [303, 304, 340, 352, 389, 401]

for i in selected:
    fp = f"dswx_data/{i:06d}.npy"
    with open(fp, "rb") as f:
        npzfile = np.load(f)
        input_data = npzfile["input"]  # (6, 512, 512) (C,H,W)
        output_data = npzfile["output"]  # (1, 512, 512) (C,H,W)
        fmask_data = npzfile["fmask"]
        sat_type = npzfile["satellite"]
    input_data = np.transpose(input_data, (1,2,0))  # (512, 512, 6) (H,W,C)
    inp_data = prepare_inp_data(input_data)   # size: (6,512,512)
    inp_data = torch.unsqueeze(inp_data, dim=0) # to have batch size of 1
    with torch.no_grad():
        test_pred, feat = model(inp_data, feat=True)
    masks = {}
    for t in tasks_outputs.keys():
        pred_img = test_pred[t][0,:,:].detach().cpu().numpy()
        thresh = optim_threshes[t]
        masks[t] = (pred_img > thresh).astype(int).squeeze()

    # 1=open water; 2=partial water; 252: snow/ice; 253: Cloud/Cloud Shadow and adjacent to cloud/cloud shadow; 254: Ocean Masked -
    tmp = np.where(output_data==1, 1, 0) + np.where(output_data==2, 1, 0) + np.where(output_data==254, 1, 0)
    water_gt = np.where(tmp>0, 1, 0)
    water_gt = water_gt.squeeze()

    fmask_water_init = ((fmask_data>>5)&1)     # Bit 5 is water
    fmask_snowice = ((fmask_data>>4)&1)
    fmask_cloudshadow = ((fmask_data>>3)&1)
    fmask_cloud = ((fmask_data>>1)&1)
    fmask_water = (fmask_water_init * (1-fmask_snowice) * (1-fmask_cloudshadow) * (1-fmask_cloud)).squeeze()

    water_mask = masks["water_mask"]
    cloudshadow_mask = masks["cloudshadow_mask"]
    cloud_mask = masks["cloud_mask"]
    snowice_mask = masks["snowice_mask"]
    sun_mask = masks["sun_mask"]
    filtered_water_mask = water_mask * (1-cloudshadow_mask) * (1-cloud_mask) * (1-snowice_mask) * sun_mask
    filtered_water_mask = filtered_water_mask.squeeze()

    _,_,_, model_TP, model_TP_FN, model_TP_FP = compute_mask_metrics(filtered_water_mask, water_gt, thresh=0.5)
    _,_,_, fmask_TP, fmask_TP_FN, fmask_TP_FP = compute_mask_metrics(fmask_water, water_gt, thresh=0.5)
    model_recall = model_TP / (model_TP_FN + EPS)
    model_precision = model_TP / (model_TP_FP + EPS)
    model_f1 = 2 * ((model_precision * model_recall) / (model_precision + model_recall + EPS))

    fmask_recall = fmask_TP / (fmask_TP_FN + EPS)
    fmask_precision = fmask_TP / (fmask_TP_FP + EPS)
    fmask_f1 = 2 * ((fmask_precision * fmask_recall) / (fmask_precision + fmask_recall + EPS))

    logger.debug(f"i: {i}, model: {model_f1}. fmask: {fmask_f1}")


    tmp = np.where(output_data==1, 1, 0) + np.where(output_data==2, 1, 0) + np.where(output_data==254, 1, 0)
    water_gt = np.where(tmp>0, 1, 0)
    out_3channel_gt = np.stack((water_gt.squeeze(),water_gt.squeeze(),water_gt.squeeze()), axis=-1)
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_gt.png"), out_3channel_gt*255)

    out_3channel_fmask = np.stack((fmask_water.squeeze(),fmask_water.squeeze(),fmask_water.squeeze()), axis=-1)
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_fmask.png"), out_3channel_fmask*255)

    out_3channel_pred = np.stack((filtered_water_mask,filtered_water_mask,filtered_water_mask), axis=-1)
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_{backbone}.png"), out_3channel_pred*255)

    nodata_val = -9999
    bgr = input_data[:,:,:3]
    feats_tmp = np.where(bgr==nodata_val, np.nanmax(bgr), bgr)
    feats_min = np.nanmin(feats_tmp)
    feats_tmp = np.where(bgr==nodata_val, feats_min, bgr)    # replace nodata values with minimum
    bgr = np.where(np.isnan(feats_tmp), feats_min, feats_tmp)    # replace nan values with minimum
    rgb = bgr[:,:,::-1]
    rgb = (rgb-np.min(rgb, axis=(0,1)))/(np.max(rgb, axis=(0,1))-np.min(rgb, axis=(0,1)))
    bgr = rgb[:,:,::-1]
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_rgb.png"), bgr*255)
    
    
    masked_bgr_fmask = np.where(out_3channel_fmask>0, 1, bgr)   # make water white
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_fmask_overlay.png"), masked_bgr_fmask*255)

    masked_bgr_pred = np.where(out_3channel_pred>0, 1, bgr)   # make water white
    cv2.imwrite(os.path.join("out_visualizations_colorado", f"{i:06d}_{backbone}_overlay.png"), masked_bgr_pred*255)


    filenames = [
        os.path.join("out_visualizations_colorado", f"{i:06d}_{backbone}_overlay.png"),
        os.path.join("out_visualizations_colorado", f"{i:06d}_rgb.png"),
    ]
    images=[]
    for filename in filenames:
        images.append(imageio.imread(filename))
    imageio.mimsave(os.path.join("out_visualizations_colorado", f"{i:06d}_{backbone}.gif"), images, duration=2000, loop=0)



    filenames = [
        os.path.join("out_visualizations_colorado", f"{i:06d}_fmask_overlay.png"),
        os.path.join("out_visualizations_colorado", f"{i:06d}_rgb.png"),
    ]
    images=[]
    for filename in filenames:
        images.append(imageio.imread(filename))
    imageio.mimsave(os.path.join("out_visualizations_colorado", f"{i:06d}_fmask.gif"), images, duration=2000, loop=0)


2024-02-07 16:28:26.794 | DEBUG    | __main__:<module>:53 - i: 303, model: 0.7787352021978013. fmask: 0.7817029782407299
/tmp/ipykernel_2598408/2200862126.py:92: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(filename))
/tmp/ipykernel_2598408/2200862126.py:103: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(filename))
2024-02-07 16:28:29.518 | DEBUG    | __main__:<module>:53 - i: 304, model: 0.8117248663667369. fmask: 0.7311202852415449
2024-02-07 16:28:32.063 | DEBUG    | __main__:<module>:53 - i: 340, model: 0.622129386642031.

In [8]:
# Generate samples for colorado